# NB-v6 — LGMF with U-Net decoder (head-to-head vs unet_drop)

The capacity test. `unet_drop` beat LGMF (both 0.801 vs 0.685) — hypothesis: the
lightweight MLP head, not the fusion idea, is the gap. This trains the **full
LGMF** (gate + FiLM + consistency + curriculum, all intact) but with a proper
**skip-connection U-Net decoder** (`decoder='unet'`), 3 seeds, same recipe.

Decision rule when it finishes:
- If LGMF-v6 matches/beats unet_drop on 'both' AND keeps a smaller both->s1 drop -> **new paper model, proceed to NB4.**
- If it still trails -> the gap is conceptual, and we pivot the framing (curriculum-as-contribution).

**Attach:** `lgmf_c` (v6), Sen1Floods11 imagery, `lgmf-index`. Internet On, GPU on.
~1.5h (3 seeds). unet decoder is ~4.6M params / 16 GFLOPs (still < unet_drop's 6.6M params).


In [ ]:
import sys, os, glob, json, random, time, warnings
warnings.filterwarnings("ignore")
init = glob.glob("/kaggle/input/**/lgmf/__init__.py", recursive=True)
assert init, "attach lgmf_c v6"
sys.path.insert(0, os.path.dirname(os.path.dirname(init[0])))
import numpy as np, torch
from torch.utils.data import DataLoader
import lgmf
from lgmf.config import LGMFConfig
from lgmf.dataset import Sen1Floods11
from lgmf.model import build_model, count_params
from lgmf.metrics import flops_g
from lgmf.ema import ModelEMA
from lgmf.engine import train_one_epoch, evaluate
print("lgmf", lgmf.__version__)
assert lgmf.__version__ >= "6.0.0", "attach lgmf_c v6"
idx = os.path.dirname(glob.glob("/kaggle/input/**/norm_stats.json", recursive=True)[0])
stats = json.load(open(f"{idx}/norm_stats.json"))

In [ ]:
SEEDS, EPOCHS = [42, 1, 2], 45
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pin = device.type == "cuda"; regs = ["both", "s1_only", "s2_only"]

def make_cfg(seed):
    c = LGMFConfig()
    c.epochs, c.lr, c.batch_size = EPOCHS, 3e-4, 8
    c.ema_decay, c.loss_type, c.stem_stride = 0.99, "focal_tversky", 2
    c.decoder = "unet"                                # <-- the v6 change
    c.num_workers = 0
    c.split_dir, c.ckpt_dir = idx, "/kaggle/working/ckpt_v6"
    c.s1_mean, c.s1_std = stats["s1_mean"], stats["s1_std"]
    c.s2_mean, c.s2_std = stats["s2_mean"], stats["s2_std"]
    c.seed, c.run_name = seed, "lgmf_v6"
    return c

def seed_all(sd):
    random.seed(sd); np.random.seed(sd); torch.manual_seed(sd); torch.cuda.manual_seed_all(sd)
    torch.backends.cudnn.benchmark = False

def run_one(cfg):
    seed_all(cfg.seed)
    tr = DataLoader(Sen1Floods11(cfg, f"{idx}/train_index.json", augment=True),
                    batch_size=cfg.batch_size, shuffle=True, num_workers=0, pin_memory=pin, drop_last=True)
    va = DataLoader(Sen1Floods11(cfg, f"{idx}/val_index.json", augment=False),
                    batch_size=cfg.batch_size, shuffle=False, num_workers=0, pin_memory=pin)
    model = build_model(cfg).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg.epochs)
    scaler = torch.amp.GradScaler("cuda") if (cfg.amp and device.type=="cuda") else None
    ema = ModelEMA(model, cfg.ema_decay); em = ema.module
    best, bv = -1.0, None
    for epoch in range(cfg.epochs):
        train_one_epoch(model, tr, opt, cfg, epoch, device, scaler=scaler, ema=ema)
        sched.step()
        val = evaluate(em, va, cfg, device)
        sc = float(np.mean([val[r]["iou"] for r in regs]))
        if sc > best:
            best = sc
            bv = {r: {"iou": val[r]["iou"], "f1": val[r]["f1"],
                      "flood_only_iou": val[r].get("flood_only_iou")} for r in regs}
            D = os.path.join(cfg.ckpt_dir, "lgmf_v6", f"seed{cfg.seed}"); os.makedirs(D, exist_ok=True)
            torch.save({"model": model.state_dict(), "ema": ema.module.state_dict(),
                        "epoch": epoch, "val": bv, "cfg": dict(cfg.__dict__)},
                       os.path.join(D, "best.pt"))
    return best, bv

## Train 3 seeds

In [ ]:
print(f"model: {count_params(build_model(make_cfg(42))):.2f}M params\n")
runs, t0 = [], time.time()
for sd in SEEDS:
    D = os.path.join("/kaggle/working/ckpt_v6", "lgmf_v6", f"seed{sd}")
    rj = os.path.join(D, "result.json")
    if os.path.exists(rj):
        rec = json.load(open(rj)); best, bv = rec["best"], rec["val"]
        print(f"[skip] seed{sd} mean {best:.3f}")
    else:
        best, bv = run_one(make_cfg(sd))
        os.makedirs(D, exist_ok=True); json.dump({"best": best, "val": bv}, open(rj,"w"), indent=2, default=str)
        print(f"[done] seed{sd} mean {best:.3f}  both {bv['both']['iou']:.3f} "
              f"s1 {bv['s1_only']['iou']:.3f} s2 {bv['s2_only']['iou']:.3f}  ({(time.time()-t0)/60:.0f} min)")
    runs.append(bv)

## Head-to-head verdict

In [ ]:
def ms(reg): a = np.array([r[reg]["iou"] for r in runs]); return a.mean(), a.std()
b, s1, s2 = ms("both"), ms("s1_only"), ms("s2_only")
drop = b[0] - s1[0]

# reference numbers from NB3 (unet_drop, 3 seeds) and NB2c (LGMF v4 / mlp head)
UNET_DROP = {"both": 0.801, "s1": 0.598, "s2": 0.781, "drop": 0.202}
LGMF_V4   = {"both": 0.685, "s1": 0.531, "s2": 0.607, "drop": 0.154}

print("=" * 68)
print(f"{'model':16s} {'both':>9s} {'s1_only':>9s} {'s2_only':>9s} {'drop b->s1':>11s}")
print("-" * 68)
print(f"{'LGMF v4 (mlp)':16s} {LGMF_V4['both']:>9.3f} {LGMF_V4['s1']:>9.3f} {LGMF_V4['s2']:>9.3f} {LGMF_V4['drop']:>11.3f}")
print(f"{'unet_drop':16s} {UNET_DROP['both']:>9.3f} {UNET_DROP['s1']:>9.3f} {UNET_DROP['s2']:>9.3f} {UNET_DROP['drop']:>11.3f}")
print(f"{'LGMF v6 (unet)':16s} {b[0]:>9.3f} {s1[0]:>9.3f} {s2[0]:>9.3f} {drop:>11.3f}")
print("=" * 68)
print(f"LGMF v6:  both {b[0]:.3f}±{b[1]:.3f}  s1 {s1[0]:.3f}±{s1[1]:.3f}  s2 {s2[0]:.3f}±{s2[1]:.3f}")
won_both = b[0] >= UNET_DROP['both'] - 0.01
smaller_drop = drop <= UNET_DROP['drop']
print()
print("VERDICT:")
print(f"  matches/beats unet_drop on 'both': {'YES' if won_both else 'NO'} ({b[0]:.3f} vs {UNET_DROP['both']:.3f})")
print(f"  smaller degradation drop:          {'YES' if smaller_drop else 'NO'} ({drop:.3f} vs {UNET_DROP['drop']:.3f})")
if won_both and smaller_drop:
    print("  -> LGMF v6 is the paper model. Proceed to NB4.")
elif b[0] >= UNET_DROP['both'] - 0.03:
    print("  -> close on 'both' + graceful-degradation edge = defensible. Proceed to NB4, lead with the drop.")
else:
    print("  -> still trails: gap is conceptual. Consider pivoting to curriculum-as-contribution.")